In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

BASE_URL = "https://api.exchange.coinbase.com"
PRODUCT_ID = "BTC-USD"
GRANULARITY = 86400  
MAX_DAYS = 300       

def iso(dt):
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")

def fetch_candles(start, end):
    url = f"{BASE_URL}/products/{PRODUCT_ID}/candles"
    params = {
        "start": iso(start),
        "end": iso(end),
        "granularity": GRANULARITY
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()


start_date = datetime(2016, 1, 31, tzinfo=timezone.utc)
end_date = datetime(2026, 2, 1, tzinfo=timezone.utc)
all_candles = []
t = start_date

while t < end_date:
    t_next = min(t + timedelta(days=MAX_DAYS), end_date)
    candles = fetch_candles(t, t_next)
    all_candles.extend(candles)
    t = t_next


df = pd.DataFrame(
    all_candles,
    columns=["timestamp", "low", "high", "open", "close", "volume"]
)

df["date"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.date
df = df.drop_duplicates(subset="date")
df = df.sort_values("date")
df = df[["date", "open", "high", "low", "close", "volume"]]


In [2]:
print(df.tail())
print(df.head())
print(f"Total rows: {len(df)}")
df.to_csv("btc_usd_daily.csv", index=False)

            date      open      high       low     close        volume
3616  2026-01-28  89116.96  90476.81  88706.32  89162.40   6636.136099
3615  2026-01-29  89162.40  89210.16  83216.21  84513.20  14860.940293
3614  2026-01-30  84513.19  84599.00  81000.12  84110.99  15232.811992
3613  2026-01-31  84110.99  84138.00  75644.15  78648.00  14855.073443
3612  2026-02-01  78648.00  79337.99  75612.75  76895.53  11545.987147
           date    open    high     low   close       volume
300  2016-01-31  378.46  382.50  365.00  367.95  5506.776813
299  2016-02-01  367.89  379.00  366.26  371.33  7931.229229
298  2016-02-02  371.33  374.41  371.17  372.93  6856.218158
297  2016-02-03  372.93  373.01  365.63  368.87  7147.956573
296  2016-02-04  368.87  390.63  368.74  387.99  8887.757232
Total rows: 3655
